# 06 — SCL Join: NRC VAD v2 + SCL-OPP + SCL-NMA

Full outer join of three sentiment lexicons on `term`, with a phrase-type router
that selects the most specific valence score available.

## Lexicon sources

| source | entries | specialisation | scale |
|--------|---------|----------------|-------|
| SCL-OPP | 1,178 | opposing-polarity phrases (e.g. `happy accident`, `bad luck`) | bipolar [-1, +1] |
| SCL-NMA | 3,207 | modifier constructions — negators, modals, adverbs (e.g. `not bad`, `barely good`) | bipolar [-1, +1] |
| NRC VAD v2.1 | 54,801 | general lexical affect; unigrams + bigrams; valence/arousal/dominance | bipolar [-1, +1] |

All three lexicons use the **same bipolar scale** (-1 = maximally negative, 0 = neutral, +1 = maximally positive).
Scores are produced via Best-Worst Scaling crowdsourcing and are directly comparable.

## Valence hierarchy (specificity beats generality)

```
1. OPP   — exact phrase match in opposing-polarity lexicon
2. NMA   — exact phrase match in negator/modal/adverb lexicon
3. VAD   — general lexical affect fallback
```

OPP and NMA are both specialised phrase-level resources and take precedence over VAD.
OPP wins over NMA when a term appears in both (opposing-polarity is the most specific signal).
The selected score is stored in `valence`; the source is stored in `valence_source`.

## Output columns

| column | description |
|--------|-------------|
| `term` | lexicon entry |
| `ngrams` | word count |
| `in_vad`, `in_opp`, `in_nma` | membership flags per lexicon |
| `vad_valence`, `vad_arousal`, `vad_dominance` | NRC VAD v2 scores |
| `opp_valence`, `opp_pos`, `opp_freq`, `opp_is_hashtag` | SCL-OPP scores + metadata |
| `nma_valence` | SCL-NMA score |
| `valence_count` | number of lexicons with a valence score |
| `valence_variance` | variance across available valence scores (null if only 1 source) |
| `valence_source` | which lexicon the selected valence comes from (`opp`/`nma`/`vad`) |
| `valence` | selected valence score following the hierarchy above; always bipolar [-1, +1] |
| `negators_contained`, `modals_contained`, `adverbs_contained` | modifier flags |

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path('data')

VAD_FILE  = DATA / 'NRC-VAD-Lexicon-v2.1/NRC-VAD-Lexicon-v2.1.txt'
OPP_FILE  = DATA / 'SCL-OPP/SCL-OPP/SCL-OPP.txt'
NMA_FILE  = DATA / 'SCL-NMA/SCL-NMA/SCL-NMA.txt'
NEG_FILE  = DATA / 'SCL-NMA/SCL-NMA/list-English-negators.txt'
MOD_FILE  = DATA / 'SCL-NMA/SCL-NMA/list-English-modals.txt'
ADV_FILE  = DATA / 'SCL-NMA/SCL-NMA/list-English-adverbs.txt'

## Load each lexicon

In [2]:
# NRC VAD v2.1 — has header row
vad = pd.read_csv(VAD_FILE, sep='\t')
vad.columns = ['term', 'vad_valence', 'vad_arousal', 'vad_dominance']
print(f'VAD: {len(vad):,} entries  range: [{vad["vad_valence"].min()}, {vad["vad_valence"].max()}]')
vad.head(3)

VAD: 54,801 entries  range: [-1.0, 1.0]


,term,vad_valence,vad_arousal,vad_dominance
0,a battery,0.134,-0.298,-0.096
1,a bit,-0.096,-0.264,-0.214
2,a bunch,0.088,-0.350,-0.068


In [3]:
# SCL-OPP — 4 columns, no header
opp = pd.read_csv(OPP_FILE, sep='\t', header=None,
                  names=['term', 'opp_valence', 'opp_pos', 'opp_freq'])

# Flag hashtag terms (term contains '#') then strip all '#' characters
opp['opp_is_hashtag'] = opp['term'].str.contains('#', regex=False)
opp['term'] = opp['term'].str.replace('#', '', regex=False).str.strip()

print(f'SCL-OPP: {len(opp):,} entries  range: [{opp["opp_valence"].min()}, {opp["opp_valence"].max()}]  ({opp["opp_is_hashtag"].sum()} hashtag terms stripped)')
opp[opp['opp_is_hashtag']].head(5)

SCL-OPP: 1,178 entries  range: [-1.0, 1.0]  (12 hashtag terms stripped)


,term,opp_valence,opp_pos,opp_freq,opp_is_hashtag
9,smile,0.953,#,57000,True
64,leave a smile,0.766,V+D+N,37,True
304,finallyinthehotel exhausted,0.297,#+#,37,True
352,wantit,0.234,#,44,True
377,dog,0.219,#,8388,True


In [4]:
# SCL-NMA — 2 columns, no header
nma = pd.read_csv(NMA_FILE, sep='\t', header=None,
                  names=['term', 'nma_valence'])
print(f'SCL-NMA: {len(nma):,} entries  range: [{nma["nma_valence"].min()}, {nma["nma_valence"].max()}]')
nma.head(3)

SCL-NMA: 3,207 entries  range: [-0.986, 0.986]


,term,nma_valence
0,enthusiastic,0.986
1,excellence,0.984
2,greatness,0.972


## Full outer join

In [5]:
joined = (
    vad
    .merge(opp, on='term', how='outer')
    .merge(nma, on='term', how='outer')
)

# ngrams: word count of term
joined['ngrams'] = joined['term'].str.split().str.len()

# membership flags
joined['in_vad'] = joined['vad_valence'].notna()
joined['in_opp'] = joined['opp_valence'].notna()
joined['in_nma'] = joined['nma_valence'].notna()

# valence_count and variance
joined['valence_count'] = joined['in_vad'].astype(int) + joined['in_opp'].astype(int) + joined['in_nma'].astype(int)
valence_cols = ['vad_valence', 'opp_valence', 'nma_valence']
joined['valence_variance'] = joined[valence_cols].var(axis=1, ddof=1)

# valence_source: OPP > NMA > VAD
joined['valence_source'] = (
    joined['in_opp'].map({True: 'opp', False: None})
    .fillna(joined['in_nma'].map({True: 'nma', False: None}))
    .fillna(joined['in_vad'].map({True: 'vad', False: None}))
)

# valence: selected score from winning source — always bipolar [-1, +1]
joined['valence'] = (
    joined['opp_valence']
    .where(joined['in_opp'], other=joined['nma_valence'])
    .where(joined['in_opp'] | joined['in_nma'], other=joined['vad_valence'])
)

# guard: assert bipolar scale [-1, +1]
assert joined['valence'].dropna().between(-1, 1).all(), \
    f"valence out of [-1, 1]: {joined['valence'].agg(['min', 'max'])}"
print('valence scale check passed: all values in [-1, +1]')

# reorder columns
joined = joined[[
    'term', 'ngrams',
    'in_vad', 'in_opp', 'in_nma',
    'vad_valence', 'vad_arousal', 'vad_dominance',
    'opp_valence', 'opp_pos', 'opp_freq', 'opp_is_hashtag',
    'nma_valence',
    'valence_count', 'valence_variance',
    'valence_source', 'valence',
]]

print(f'Joined: {len(joined):,} rows')
print(f'valence range: [{joined["valence"].min():.4f}, {joined["valence"].max():.4f}]')
print(f'\nvalence_source breakdown:')
print(joined['valence_source'].value_counts().to_string())
print(f'\nLexicon membership:')
print(f'  in_vad only:       {(joined["in_vad"] & ~joined["in_opp"] & ~joined["in_nma"]).sum():,}')
print(f'  in_opp only:       {(~joined["in_vad"] & joined["in_opp"] & ~joined["in_nma"]).sum():,}')
print(f'  in_nma only:       {(~joined["in_vad"] & ~joined["in_opp"] & joined["in_nma"]).sum():,}')
print(f'  in_vad + in_opp:   {(joined["in_vad"] & joined["in_opp"] & ~joined["in_nma"]).sum():,}')
print(f'  in_vad + in_nma:   {(joined["in_vad"] & ~joined["in_opp"] & joined["in_nma"]).sum():,}')
print(f'  in_opp + in_nma:   {(~joined["in_vad"] & joined["in_opp"] & joined["in_nma"]).sum():,}')
print(f'  in all three:      {(joined["in_vad"] & joined["in_opp"] & joined["in_nma"]).sum():,}')
joined.head(5)

valence scale check passed: all values in [-1, +1]
Joined: 57,005 rows
valence range: [-1.0000, 1.0000]

valence_source breakdown:
valence_source
vad    52824
nma     3003
opp     1178

Lexicon membership:
  in_vad only:       52,824
  in_opp only:       611
  in_nma only:       1,587
  in_vad + in_opp:   361
  in_vad + in_nma:   1,416
  in_opp + in_nma:   3
  in all three:      203


,term,ngrams,in_vad,in_opp,in_nma,vad_valence,vad_arousal,vad_dominance,opp_valence,opp_pos,opp_freq,opp_is_hashtag,nma_valence,valence_count,valence_variance,valence_source,valence
0,3rd degree burns,3.0,False,True,False,NaN,NaN,NaN,-0.828,A+N+N,11.0,False,NaN,1,NaN,opp,-0.828
1,FALSE,1.0,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.444,1,NaN,nma,-0.444
2,TRUE,1.0,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.625,1,NaN,nma,0.625
3,a bad deal,3.0,False,True,False,NaN,NaN,NaN,-0.594,D+A+N,13.0,False,NaN,1,NaN,opp,-0.594
4,a battery,2.0,True,False,False,0.134,-0.298,-0.096,NaN,NaN,NaN,NaN,NaN,1,NaN,vad,0.134


## Valence count & variance

In [6]:
print('Terms by valence_count:')
print(joined['valence_count'].value_counts().sort_index().to_string())

# variance is only meaningful when valence_count >= 2
multi = joined[joined['valence_count'] >= 2]
print(f'\nTerms with 2+ valence scores: {len(multi):,}')
print(f'valence_variance stats:')
print(multi['valence_variance'].describe().round(4).to_string())

print(f'\nHigh variance (>0.1) — lexicons disagree most:')
multi[multi['valence_variance'] > 0.1].sort_values('valence_variance', ascending=False)[
    ['term', 'valence_source', 'valence', 'vad_valence', 'opp_valence', 'nma_valence', 'valence_count', 'valence_variance']
].head(20)

Terms by valence_count:
valence_count
1    55022
2     1780
3      203

Terms with 2+ valence scores: 1,983
valence_variance stats:
count    1983.0000
mean        0.0329
std         0.0482
min         0.0000
25%         0.0035
50%         0.0142
75%         0.0423
max         0.4122

High variance (>0.1) — lexicons disagree most:


,term,valence_source,valence,vad_valence,opp_valence,nma_valence,valence_count,valence_variance
7848,chill,opp,-0.234,0.674,-0.234,NaN,2,0.412232
47320,staunchness,nma,-0.194,0.667,NaN,-0.194,2,0.370661
39221,propitious,nma,0.032,0.889,NaN,0.032,2,0.367225
21151,goodbye,nma,0.319,-0.520,NaN,0.319,2,0.351960
1430,ambiguity,nma,0.153,-0.672,NaN,0.153,2,0.340313
47945,stroke,opp,-0.688,0.124,-0.688,NaN,2,0.329672
27478,justifiably,nma,0.194,1.000,NaN,0.194,2,0.324818
51517,true,opp,0.000,0.792,0.000,NaN,2,0.313632
5421,bored,opp,0.094,-0.694,0.094,NaN,2,0.310472
2323,argument,nma,-0.762,0.020,NaN,-0.762,2,0.305762


## N-gram distribution by source

In [7]:
def ngram_counts(df, label):
    counts = df['ngrams'].value_counts().sort_index()
    counts.index = counts.index.map({1: 'unigram', 2: 'bigram', 3: 'trigram'}).fillna('4+')
    print(f'\n{label}')
    print(counts.to_string())

ngram_counts(joined[joined['in_vad']], 'NRC VAD v2')
ngram_counts(joined[joined['in_opp']], 'SCL-OPP')
ngram_counts(joined[joined['in_nma']], 'SCL-NMA')
ngram_counts(joined, 'Full join')


NRC VAD v2
ngrams
unigram    44730
bigram     10071
trigram        2

SCL-OPP
ngrams
unigram    602
bigram     311
trigram    265

SCL-NMA
ngrams
unigram    1623
bigram      922
trigram     605
4+           59

Full join
ngrams
unigram    44814
bigram     11259
trigram      872
4+            59


## Modifier flags: negators / modals / adverbs contained

In [8]:
negators = [l.strip() for l in NEG_FILE.read_text().splitlines() if l.strip()]
modals   = [l.strip() for l in MOD_FILE.read_text().splitlines() if l.strip()]
adverbs  = [l.strip() for l in ADV_FILE.read_text().splitlines() if l.strip()]

print(f'negators: {negators}')
print(f'modals:   {modals}')
print(f'adverbs:  {adverbs}')

def find_contained(term, word_list):
    """Return space-separated list of modifier tokens found in term (word-boundary match), or None."""
    tokens = str(term).lower().split()
    found = []
    for modifier in word_list:
        mod_tokens = modifier.split()
        for i in range(len(tokens) - len(mod_tokens) + 1):
            if tokens[i:i + len(mod_tokens)] == mod_tokens:
                found.append(modifier)
                break
    return ' '.join(found) if found else None

joined['negators_contained'] = joined['term'].apply(find_contained, word_list=negators)
joined['modals_contained']   = joined['term'].apply(find_contained, word_list=modals)
joined['adverbs_contained']  = joined['term'].apply(find_contained, word_list=adverbs)

print(f'\nTerms with negators: {joined["negators_contained"].notna().sum():,}')
print(f'Terms with modals:   {joined["modals_contained"].notna().sum():,}')
print(f'Terms with adverbs:  {joined["adverbs_contained"].notna().sum():,}')

negators: ['cannot', 'could not', 'did not', 'does not', 'had no', 'have no', 'may not', 'never', 'no', 'not', 'nothing', 'was no', 'was not', 'will not', 'would not']
modals:   ['can', 'could', 'may', 'might', 'must', 'should', 'would']
adverbs:  ['certainly', 'especially', 'extremely', 'fairly', 'highly', 'increasingly', 'less', 'more', 'most', 'much', 'much more', 'particularly', 'pretty', 'probably', 'quite', 'rather', 'really', 'relatively', 'so', 'too', 'very', 'very very']



Terms with negators: 547
Terms with modals:   515
Terms with adverbs:  880


In [9]:
print(f'Final shape: {joined.shape}')
print(f'Columns: {list(joined.columns)}')
joined.sample(10)

Final shape: (57005, 20)
Columns: ['term', 'ngrams', 'in_vad', 'in_opp', 'in_nma', 'vad_valence', 'vad_arousal', 'vad_dominance', 'opp_valence', 'opp_pos', 'opp_freq', 'opp_is_hashtag', 'nma_valence', 'valence_count', 'valence_variance', 'valence_source', 'valence', 'negators_contained', 'modals_contained', 'adverbs_contained']


,term,ngrams,in_vad,in_opp,in_nma,vad_valence,vad_arousal,vad_dominance,opp_valence,opp_pos,opp_freq,opp_is_hashtag,nma_valence,valence_count,valence_variance,valence_source,valence,negators_contained,modals_contained,adverbs_contained
44239,serenely,1.0,True,False,False,1.000,-1.000,0.381,NaN,NaN,NaN,NaN,NaN,1,NaN,vad,1.000,NaN,NaN,NaN
29604,lucky dog,2.0,True,False,False,0.566,0.018,0.412,NaN,NaN,NaN,NaN,NaN,1,NaN,vad,0.566,NaN,NaN,NaN
48523,sunstroke,1.0,True,False,False,-0.667,0.708,-0.556,NaN,NaN,NaN,NaN,NaN,1,NaN,vad,-0.667,NaN,NaN,NaN
48334,succumb,1.0,True,False,False,-0.596,0.574,-0.416,NaN,NaN,NaN,NaN,NaN,1,NaN,vad,-0.596,NaN,NaN,NaN
19642,freshen,1.0,True,False,False,0.612,-0.274,0.132,NaN,NaN,NaN,NaN,NaN,1,NaN,vad,0.612,NaN,NaN,NaN
4632,biosynthesis,1.0,True,False,False,0.667,0.667,0.429,NaN,NaN,NaN,NaN,NaN,1,NaN,vad,0.667,NaN,NaN,NaN
34060,not wrong,2.0,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1,NaN,nma,0.000,not,NaN,NaN
56017,wine bottle,2.0,True,False,False,0.096,-0.114,-0.108,NaN,NaN,NaN,NaN,NaN,1,NaN,vad,0.096,NaN,NaN,NaN
21863,gullible,1.0,True,False,False,-0.120,-0.116,-0.364,NaN,NaN,NaN,NaN,NaN,1,NaN,vad,-0.120,NaN,NaN,NaN
54130,variant,1.0,True,False,False,-0.296,0.072,-0.068,NaN,NaN,NaN,NaN,NaN,1,NaN,vad,-0.296,NaN,NaN,NaN


## Audit: constituent coverage

For every bigram and trigram in the joined dataset, check:
- Do its **unigram tokens** appear as entries in the dataset?
- For trigrams: do its **sub-bigrams** (token[0:2], token[1:3]) appear as bigram entries?

This tells us whether the lexicons are self-consistent — i.e. whether we can always
fall back to constituent lookups when a phrase is missing.

In [10]:
term_set = set(joined['term'].dropna().str.lower())

def check_constituents(term):
    tokens = term.lower().split()
    n = len(tokens)
    if n < 2:
        return None
    result = {'term': term, 'ngrams': n}
    result['unigram_tokens']   = tokens
    result['unigrams_in_set']  = [t for t in tokens if t in term_set]
    result['unigrams_missing'] = [t for t in tokens if t not in term_set]
    if n == 3:
        sub_bigrams = [' '.join(tokens[0:2]), ' '.join(tokens[1:3])]
        result['sub_bigrams']         = sub_bigrams
        result['sub_bigrams_in_set']  = [b for b in sub_bigrams if b in term_set]
        result['sub_bigrams_missing'] = [b for b in sub_bigrams if b not in term_set]
    return result

phrases = joined[joined['ngrams'] >= 2]['term'].dropna().tolist()
audit_rows = [check_constituents(t) for t in phrases]
audit = pd.DataFrame(audit_rows)

print(f'Phrases audited: {len(audit):,}')
audit['all_unigrams_covered'] = audit['unigrams_missing'].apply(lambda x: len(x) == 0)
print(f'\nUnigram coverage:')
print(audit['all_unigrams_covered'].value_counts().to_string())

trigram_audit = audit[audit['ngrams'] == 3].copy()
if len(trigram_audit):
    trigram_audit['all_sub_bigrams_covered'] = trigram_audit['sub_bigrams_missing'].apply(lambda x: len(x) == 0)
    print(f'\nTrigram sub-bigram coverage:')
    print(trigram_audit['all_sub_bigrams_covered'].value_counts().to_string())

Phrases audited: 12,190

Unigram coverage:
all_unigrams_covered
True     10085
False     2105

Trigram sub-bigram coverage:
all_sub_bigrams_covered
False    837
True      35


In [11]:
missing = audit[~audit['all_unigrams_covered']]
print(f'Phrases with at least one unigram token NOT in the dataset: {len(missing):,}')
missing[['term', 'ngrams', 'unigrams_missing']].head(20)

Phrases with at least one unigram token NOT in the dataset: 2,105


,term,ngrams,unigrams_missing
0,3rd degree burns,3,[3rd]
1,a bad deal,3,[a]
2,a battery,2,[a]
3,a bit,2,[a]
4,a bunch,2,[a]
5,a cappella,2,"[a, cappella]"
6,a couple,2,[a]
7,a crazy year,3,[a]
8,a drive,2,[a]
9,a few,2,[a]


In [12]:
if len(trigram_audit):
    missing_bigrams = trigram_audit[trigram_audit['sub_bigrams_missing'].apply(len) > 0]
    print(f'Trigrams with missing sub-bigrams: {len(missing_bigrams):,}')
    missing_bigrams[['term', 'sub_bigrams', 'sub_bigrams_missing']].head(20)

Trigrams with missing sub-bigrams: 837


## Composition shift

For multi-word terms with a phrase-level valence (OPP or NMA), compute:

```
computed_valence  = mean(hierarchy-selected valence of each unigram token)
composition_shift = selected_valence − computed_valence
```

The shift measures **how much phrase meaning deviates from naive word averaging**.

| shift | interpretation |
|-------|----------------|
| ≈ 0 | additive — phrase means what its words mean |
| > 0 | phrase more positive than expected |
| < 0 | phrase more negative than expected (negation, irony) |
| large |abs| | strong compositional transformation |

In [13]:
# unigram valence lookup (hierarchy-selected, not raw VAD)
unigram_valence = (
    joined[(joined['ngrams'] == 1) & joined['valence'].notna()]
    .drop_duplicates('term')
    .set_index('term')['valence']
    .to_dict()
)
print(f'Unigrams available for token-mean: {len(unigram_valence):,}')

def compute_token_mean(term):
    """Mean of hierarchy-selected unigram valences for each token in term."""
    if not isinstance(term, str):
        return None
    tokens = term.lower().split()
    scores = [unigram_valence[t] for t in tokens if t in unigram_valence]
    return float(sum(scores) / len(scores)) if scores else None

# computed_valence: only meaningful for multi-word terms
joined['computed_valence'] = joined.apply(
    lambda r: compute_token_mean(r['term']) if r['ngrams'] > 1 else r['valence'],
    axis=1
)

# composition_shift: only defined where both values exist
joined['composition_shift']     = joined['valence'] - joined['computed_valence']
joined['abs_composition_shift'] = joined['composition_shift'].abs()

multi = joined[(joined['ngrams'] > 1) & joined['composition_shift'].notna()]
print(f'Multi-word terms with composition shift: {len(multi):,}')
print(f'\ncomposition_shift stats:')
print(multi['composition_shift'].describe().round(4).to_string())

print(f'\nLargest positive shifts (phrase >> words):')
print(multi.nlargest(10, 'composition_shift')[
    ['term', 'valence_source', 'valence', 'computed_valence', 'composition_shift']
].to_string(index=False))

print(f'\nLargest negative shifts (phrase << words, e.g. negation):')
print(multi.nsmallest(10, 'composition_shift')[
    ['term', 'valence_source', 'valence', 'computed_valence', 'composition_shift']
].to_string(index=False))

Unigrams available for token-mean: 44,811


Multi-word terms with composition shift: 12,085

composition_shift stats:
count    12085.0000
mean        -0.1285
std          0.3096
min         -1.7470
25%         -0.3205
50%         -0.1230
75%          0.0715
max          1.1430

Largest positive shifts (phrase >> words):


           term valence_source  valence  computed_valence  composition_shift
        no harm            nma    0.333         -0.810000           1.143000
  not dangerous            nma    0.236         -0.806000           1.042000
problem solving            vad    0.620         -0.422000           1.042000
    not trouble            nma    0.403         -0.597000           1.000000
   not negative            vad    0.244         -0.736000           0.980000
     no worries            vad    0.162         -0.812000           0.974000
awesome as hell            opp    0.922         -0.020667           0.942667
  does not hurt            nma    0.347         -0.594000           0.941000
      not worse            nma    0.194         -0.736000           0.930000
 stepping stone            vad    0.354         -0.568000           0.922000

Largest negative shifts (phrase << words, e.g. negation):
          term valence_source  valence  computed_valence  composition_shift
      not love    

In [14]:
# Save joined lexicon for use in downstream notebooks
import pickle

LEXICON_DIR = DATA / 'lexicons'
LEXICON_DIR.mkdir(parents=True, exist_ok=True)

out = LEXICON_DIR / 'scl_joined.pkl'
joined.to_pickle(out)
print(f'Saved → {out}  ({out.stat().st_size / 1e6:.1f} MB)')
print(f'Shape: {joined.shape}')
print(f'Columns: {list(joined.columns)}')

Saved → data/lexicons/scl_joined.pkl  (9.6 MB)
Shape: (57005, 23)
Columns: ['term', 'ngrams', 'in_vad', 'in_opp', 'in_nma', 'vad_valence', 'vad_arousal', 'vad_dominance', 'opp_valence', 'opp_pos', 'opp_freq', 'opp_is_hashtag', 'nma_valence', 'valence_count', 'valence_variance', 'valence_source', 'valence', 'negators_contained', 'modals_contained', 'adverbs_contained', 'computed_valence', 'composition_shift', 'abs_composition_shift']
